# Free Photo-to-Talking-Video (Hindi speech) — SadTalker

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parmarharmad-art/Social-ai/blob/main/social-content/Free_Hindi_Photo_LipSync_Colab.ipynb)

**Photo-only upload:** upload one portrait. Enter a Hindi script (or use the editable default below); the notebook creates Hindi speech with free, no-key Edge TTS, then animates the photo with open-source SadTalker.

**Output:** a talking-photo MP4 only if inference succeeds and both video and audio streams validate. This is model-based portrait animation, not a static image with voiceover. Free GPU access and third-party TTS availability are not guaranteed.

**Phone steps:** open in Colab → Runtime → Change runtime type → choose GPU if available → run all cells from top to bottom → upload only your portrait when prompted → enter/keep the Hindi script → download the MP4 at the end.

In [ ]:
import os, subprocess, sys
from pathlib import Path
print('Python:', sys.version)
from google.colab import files
uploaded = files.upload()  # Upload ONE portrait image only; narration audio is generated below.
images = [p for p in Path('/content').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
print('Image candidates:', [p.name for p in images])
assert images, 'Upload one portrait image (.jpg/.png/.webp).'
image_file = images[0].resolve()
assert image_file.parent == Path('/content') and image_file.stat().st_size > 0, 'Photo is missing or empty.'
# Edit this default, or paste your own Hindi narration when prompted.
default_hindi_script = 'नमस्ते! मैं किरण कुमार हूँ। आज मैं आपको बताऊँगा कि AI ऑटोमेशन से सोशल मीडिया का काम आसान कैसे बनाया जा सकता है। ऐसी उपयोगी टिप्स के लिए मुझे फॉलो करें।'
script = input('Hindi narration script (press Enter to use the default script): ').strip()
if not script:
    script = default_hindi_script
assert script.strip(), 'Hindi script cannot be empty.'
print('Portrait selected:', image_file.name)
print('Hindi script characters:', len(script))

## Install SadTalker
Run the next cell once. This downloads the source and model checkpoints; it may take several minutes.

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
repo = Path('/content/SadTalker')
def run_checked(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)
if repo.exists() and (repo / 'inference.py').is_file():
    print('SadTalker checkout already exists; reusing it.')
else:
    if repo.exists(): shutil.rmtree(repo)
    run_checked(['git','clone','--depth','1','https://github.com/OpenTalker/SadTalker.git',repo])
print('Installing ffmpeg...')
run_checked(['bash','-lc','apt-get update -qq && apt-get install -y -qq ffmpeg'])
print('Installing Python packages without the upstream legacy scikit-image pin or reinstalling Colab Torch...')
python_minor = sys.version_info.minor
scipy_pin = 'scipy==1.11.4' if sys.version_info >= (3,12) else 'scipy==1.10.1'
print('Python compatibility selection:', sys.version.split()[0], '|', scipy_pin)
packages = ['numpy==1.26.4','face_alignment==1.3.5','imageio==2.19.3','imageio-ffmpeg==0.4.7','librosa==0.9.2','numba','resampy==0.3.1','pydub==0.25.1',scipy_pin,'kornia==0.6.8','tqdm','yacs==0.1.8','pyyaml','joblib==1.3.2','scikit-image==0.21.0','basicsr==1.4.2','facexlib==0.3.0','gradio','gfpgan','av','safetensors','edge-tts']
run_checked([sys.executable,'-m','pip','install','-q'] + packages)
print('Installed package metadata check (diagnostic only):')
subprocess.run([sys.executable,'-m','pip','check'], check=False)
print('Patching known BasicSR/TorchVision import if present...')
import site
patch_candidates = []
for site_dir in site.getsitepackages(): patch_candidates.extend(Path(site_dir).glob('basicsr/data/degradations.py'))
for patch_file in patch_candidates:
    text = patch_file.read_text()
    old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
    if old in text:
        patch_file.write_text(text.replace(old, 'from torchvision.transforms.functional import rgb_to_grayscale'))
        print('Patched:', patch_file)
print('Downloading model checkpoints (large files; wait until command completes)...')
run_checked(['bash','scripts/download_models.sh'], cwd=repo)
checkpoint_dir = repo/'checkpoints'
needed = ['mapping_00109-model.pth.tar','mapping_00229-model.pth.tar','SadTalker_V0.0.2_256.safetensors','SadTalker_V0.0.2_512.safetensors']
missing = [name for name in needed if not (checkpoint_dir/name).is_file() or (checkpoint_dir/name).stat().st_size < 100_000]
if missing: raise RuntimeError('Missing or incomplete model checkpoints: '+', '.join(missing))
print('Setup completed and required model checkpoint files exist.')
print('Generating Hindi narration with free/no-key Edge TTS...')
import edge_tts
narration_path = Path('/content/hindi_narration.mp3')
await edge_tts.Communicate(script, 'hi-IN-SwaraNeural').save(str(narration_path))
assert narration_path.is_file() and narration_path.stat().st_size > 1000, 'Hindi TTS did not create a usable narration file.'
print('Hindi narration generated:', narration_path, '| bytes:', narration_path.stat().st_size)


In [ ]:
from pathlib import Path
import subprocess, sys, shutil
# Use the one portrait uploaded in cell 1 and narration generated in cell 3.
image_file = Path('/content') / images[0].name
audio_file = Path('/content/hindi_narration.mp3').resolve()
assert image_file.parent == Path('/content') and image_file.is_file() and image_file.stat().st_size > 0, f'Photo missing/empty or invalid path: {image_file.name}'
assert audio_file.parent == Path('/content') and audio_file.is_file() and audio_file.stat().st_size > 0, 'Generated Hindi narration is missing or empty.'
normalized_audio = Path('/content/hindi_narration_16k_mono.wav')
print('Validating portrait and normalizing Hindi audio...')
try:
    from PIL import Image
    with Image.open(image_file) as im: im.verify()
except Exception as e:
    raise RuntimeError(f'Photo could not be decoded; upload a standard JPG or PNG: {e!r}') from e
subprocess.run(['ffmpeg','-y','-hide_banner','-loglevel','error','-i',str(audio_file),'-vn','-ac','1','-ar','16000','-c:a','pcm_s16le',str(normalized_audio)], check=True)
audio_probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(normalized_audio)], check=True, capture_output=True, text=True)
duration = float(audio_probe.stdout.strip())
assert duration > 0.2, f'Audio is too short ({duration:.2f}s)'
print(f'Audio OK: {duration:.1f}s. Starting SadTalker inference; this may take several minutes...')
out_dir = Path('/content/SadTalker/results/kiran_talking_photo')
out_dir.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'inference.py', '--driven_audio', str(normalized_audio), '--source_image', str(image_file), '--result_dir', str(out_dir), '--still', '--preprocess', 'crop']
subprocess.run(cmd, cwd='/content/SadTalker', check=True)
videos = sorted((p for p in out_dir.rglob('*.mp4') if p.is_file() and p.stat().st_size > 1000), key=lambda p: p.stat().st_mtime, reverse=True)
assert videos, f'SadTalker finished without a usable MP4. Inspect the inference logs above. Output directory: {out_dir}'
output_path = Path('/content/talking_photo_hindi.mp4')
shutil.copy2(videos[0], output_path)
probe = subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type,width,height','-of','csv=p=0',str(output_path)], check=True, capture_output=True, text=True)
streams = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
assert any(line.startswith('video') for line in streams), f'Output has no video stream: {streams}'
assert any(line.startswith('audio') for line in streams), f'Output has no audio stream: {streams}'
assert output_path.stat().st_size > 1000, 'Output MP4 is empty or incomplete.'
print('SUCCESS — validated talking-photo MP4:', output_path, '| bytes:', output_path.stat().st_size)
from IPython.display import Video, display
display(Video(str(output_path), embed=True))


In [ ]:
from google.colab import files
assert Path('/content/talking_photo_hindi.mp4').is_file() and Path('/content/talking_photo_hindi.mp4').stat().st_size > 1000
files.download('/content/talking_photo_hindi.mp4')

## Troubleshooting
- `CUDA available: False` means this session has no GPU; choose Runtime → Change runtime type → GPU if offered, then rerun from the top. Free GPU access is not guaranteed.
- Setup intentionally installs compatible `scikit-image` and leaves Colab's preinstalled Torch in place instead of blindly installing every old pinned requirement. Other dependency conflicts can still occur.
- If the package installation fails, stop at that cell and copy the first red error. Do not continue to inference with a partially installed environment.
- The model download cell validates four required checkpoint files and their size. If it fails, inspect download errors; do not run inference with incomplete model files.
- If inference fails, send the final traceback. If no non-empty MP4 is found, the run is not successful.
- SadTalker is open-source, but free GPU availability, runtime duration, and hosting are not guaranteed.